In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("../data/processed/both_aligned_feature_extracted_train(4s).csv")

df["activity"] = df["activity"].astype("category")

In [4]:
X = df.drop(columns="activity")
y = df["activity"]

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, make_scorer

model = RandomForestClassifier(
    n_estimators=200,
    criterion="entropy",
    n_jobs=-1,
)


score = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring=make_scorer(f1_score, average="macro"),
    n_jobs=-1,
).mean()

score

np.float64(0.9031726708932835)

In [6]:
model.fit(X, y)
import joblib

joblib.dump(model,"../models/rfc_aligned.pkl")

['../models/rfc_aligned.pkl']

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score


def objective(trial):

    n_estimators = trial.suggest_int("n_estimators", 100, 500)

    max_depth = trial.suggest_int("max_depth", 5, 50)

    min_samples_split = trial.suggest_int("min_samples_split", 2, 5)

    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)

    criterion = trial.suggest_categorical("criterion", ["entropy", "gini"])

    max_features = trial.suggest_categorical(
        "max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]
    )

    max_samples = trial.suggest_categorical("max_samples", [0.9, 1])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        criterion=criterion,
        max_features=max_features,
        max_samples=max_samples,
        bootstrap=True,
        n_jobs=-1,
        random_state=42,
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=3,
        scoring=make_scorer(f1_score, average="macro"),
        n_jobs=-1,
    ).mean()

    return score


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(),
)


study.optimize(objective, n_trials=50)